# Residualization smoke test -- synthetic data

Exercises the full phenotype residualization pipeline (`../../scripts/local/residualize_lib.R`, the same code `residualize_phenotypes.ipynb` uses for real) against synthetic IDs, phenotypes, and PCs -- no AoU access needed. Two fake phenotypes: one generated near-normal (`fake_bmi`), one generated log-normal with injected outliers (`fake_triglycerides`), specifically so the diagnostics show whether the inverse-normal transform and outlier trimming are actually doing something.

## Setup

**Run this cell first in a freshly restarted kernel.** If a package (e.g. `rlang`) already got auto-loaded from the system library before this cell runs, `.libPaths()` can't retroactively swap it for the newer version installed here -- R won't unload/replace an already-attached namespace mid-session. Symptom: `namespace 'rlang' 1.1.6 is already loaded, but >= 1.1.7 is required`. Fix is a full kernel restart, not re-running cells in the same session.

In [ ]:
# R_LIBS_USER is set on this environment, but to a colon-separated list of
# non-writable system paths (conda + spark), not a single writable directory --
# install.packages(lib=...) needs one directory, so don't trust R_LIBS_USER here.
# Always use an explicit personal library instead.
user_lib <- file.path(Sys.getenv("HOME"), "R", "library")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

required_pkgs <- c("dplyr", "readr", "e1071")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs, lib = user_lib)

library(dplyr)
library(readr)
source("../../scripts/local/residualize_lib.R")

set.seed(42)
OUT_DIR <- "/tmp/fake_pheno_out"
unlink(OUT_DIR, recursive = TRUE)

## Synthetic inputs

200 fake person IDs, fake sex-at-birth, fake PCs (written to a file and read back, same as a real `.eigenvec`), and fake zip3/SES joined via zip3 -- same join shape the real covariates use.

In [ ]:
N <- 200
keep_ids <- sprintf("SIM%04d", 1:N)

sex_lookup <- tibble(
  person_id = keep_ids,
  sex_at_birth = sample(c("Female", "Male"), N, replace = TRUE)
)

all_pc_cols <- paste0("PC", 1:20)   # fake .eigenvec has 20 columns, same as a real one
pc_cols <- paste0("PC", 1:5)        # only the first 5 actually get used as covariates -- see residualize_phenotypes.ipynb
fake_pcs <- as.data.frame(matrix(rnorm(N * 20), nrow = N, ncol = 20))
names(fake_pcs) <- all_pc_cols
fake_pcs <- bind_cols(tibble(`#FID` = keep_ids, IID = keep_ids), fake_pcs)

pc_path <- "/tmp/fake_pcs.eigenvec"
write_tsv(fake_pcs, pc_path)

zip3_codes <- sprintf("%03d", sample(100:999, 15))
person_zip3 <- tibble(person_id = keep_ids, zip3 = sample(zip3_codes, N, replace = TRUE))
ses_by_zip3 <- tibble(
  zip3 = zip3_codes,
  median_income = round(rnorm(15, 60000, 15000)),
  poverty = pmax(0, pmin(1, rnorm(15, 0.12, 0.05))),
  deprivation_index = rnorm(15, 0, 1)
)

## Fake phenotype list + pull functions

`fake_bmi`: roughly normal, no outliers. `fake_triglycerides`: log-normal (right-skewed) with 5 injected large outliers -- deliberately picked so the diagnostics below show a real before/after difference, not just near-zero skew everywhere.

In [ ]:
pheno_list <- tibble(
  phenotype_name = c("fake_bmi", "fake_triglycerides"),
  source = c("measurement", "measurement"),
  concept_id = c("0", "0"),
  plausible_min = c(10, 10),
  plausible_max = c(60, 1000)   # generous enough that fake_triglycerides' injected
                                # +400 SD-trim outliers stay in range -- this filter
                                # is meant to catch fake_bmi's implausible values only
)

pull_phenotype_fake <- function(row, keep_ids) {
  n <- length(keep_ids)
  sex <- sex_lookup$sex_at_birth[match(keep_ids, sex_lookup$person_id)]
  age <- round(runif(n, 18, 80))
  sex_effect <- ifelse(sex == "Female", 2, -2)

  if (row$phenotype_name == "fake_bmi") {
    pheno <- 25 + sex_effect + 0.05 * age + rnorm(n, sd = 4)
    implausible_idx <- sample(n, 3)   # simulated data-entry errors (decimal-point/unit typos),
    pheno[implausible_idx] <- c(999, -12, 0.4)   # distinct from the SD-trim outliers below --
                                                  # these are implausible on their own, not just
                                                  # extreme relative to a fitted model
  } else {
    pheno <- exp(rnorm(n, mean = 4.5, sd = 0.4)) + 0.3 * age
    outlier_idx <- sample(n, 5)
    pheno[outlier_idx] <- pheno[outlier_idx] + 400
  }

  tibble(person_id = keep_ids, phenotype = pheno, age = age, sex_at_birth = sex)
}

pull_covariates_fake <- function(keep_ids) {
  pcs <- read_tsv(pc_path, show_col_types = FALSE) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids) %>%
    select(-`#FID`)

  zip3 <- person_zip3 %>% filter(person_id %in% keep_ids)
  ses <- zip3 %>% left_join(ses_by_zip3, by = "zip3") %>% select(-zip3)

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Run

Same `run_residualization()` the real notebook calls -- full cross product of {phenotype} x {raw, invnorm} x {covariate-set}, one `.pheno` file per combination.

In [ ]:
result <- run_residualization(
  pheno_list, keep_ids, pull_phenotype_fake, pull_covariates_fake,
  build_covariate_sets(pc_cols), OUT_DIR
)

## Diagnostics

`result$range_summary_table`: `fake_bmi` should show exactly 3 excluded by
`filter_plausible_range()` (the injected implausible values); `fake_triglycerides`
should show 0 -- its generated range and injected SD-trim outliers both stay
within `plausible_min`/`plausible_max`. `fake_bmi` skewness should stay near 0
either way (generated near-normal). `fake_triglycerides` raw skewness should be
clearly positive, dropping to ~0 after the invnorm transform -- if it doesn't,
something's wrong with the transform, not the fake data.

In [ ]:
result$range_summary_table   # fake_bmi should show 3 excluded, fake_triglycerides 0

In [ ]:
result$skew_summary_table

In [ ]:
result$combo_summary_table

## Spot-check the output files

In [ ]:
list.files(OUT_DIR)

In [ ]:
read.table(file.path(OUT_DIR, "fake_triglycerides__invnorm__base_pcs.pheno"), header = TRUE) %>% head()